<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/LR42_Hancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import pandas as pd
import json

with open('/content/drive/MyDrive/clinical_data.json') as f:
    clinical = pd.DataFrame(json.load(f))

with open('/content/drive/MyDrive/pathological_data.json') as f:
    pathology = pd.DataFrame(json.load(f))

df = clinical.merge(
    pathology,
    on='patient_id',
    how='inner'
)

print(df.shape)
print(df.columns.tolist())

(763, 49)
['patient_id', 'year_of_initial_diagnosis', 'age_at_initial_diagnosis', 'sex', 'smoking_status', 'primarily_metastasis', 'survival_status', 'survival_status_with_cause', 'days_to_last_information', 'first_treatment_intent', 'first_treatment_modality', 'days_to_first_treatment', 'adjuvant_treatment_intent', 'adjuvant_radiotherapy', 'adjuvant_radiotherapy_modality', 'adjuvant_systemic_therapy', 'adjuvant_systemic_therapy_modality', 'adjuvant_radiochemotherapy', 'recurrence', 'days_to_recurrence', 'progress_1', 'days_to_progress_1', 'progress_2', 'days_to_progress_2', 'metastasis_1_locations', 'days_to_metastasis_1', 'metastasis_2_locations', 'days_to_metastasis_2', 'metastasis_3_locations', 'days_to_metastasis_3', 'metastasis_4_locations', 'days_to_metastasis_4', 'primary_tumor_site', 'pT_stage', 'pN_stage', 'grading', 'hpv_association_p16', 'number_of_positive_lymph_nodes', 'number_of_resected_lymph_nodes', 'perinodal_invasion', 'lymphovascular_invasion_L', 'vascular_invasion_

In [42]:
print(df['hpv_association_p16'].value_counts(dropna=False))

hpv_association_p16
not_tested    431
negative      191
positive      141
Name: count, dtype: int64


In [43]:
df = df[
    df['hpv_association_p16'].isin(
        ['positive','negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative':0,
    'positive':1
})

print(df['HPV'].value_counts())

HPV
0    191
1    141
Name: count, dtype: int64


In [44]:
from sklearn.model_selection import train_test_split

In [45]:
df = df[
    df['hpv_association_p16'].isin(
        ['positive','negative']
    )
].copy()

df['HPV'] = df['hpv_association_p16'].map({
    'negative':0,
    'positive':1
})

features = [
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'grading'
]

X = df[features]
y = df['HPV']

In [46]:
print(df['sex'].value_counts(dropna=False))
print(df['smoking_status'].value_counts(dropna=False))
print(df['pT_stage'].value_counts(dropna=False))
print(df['pN_stage'].value_counts(dropna=False))

sex
male      255
female     77
Name: count, dtype: int64
smoking_status
smoker        154
non-smoker     87
former         86
None            5
Name: count, dtype: int64
pT_stage
pT2     141
pT1     108
pT3      66
pT4a     13
TX        2
pT4b      2
Name: count, dtype: int64
pN_stage
pN0     77
pN1     75
pN2b    54
pN2     36
NX      27
pN2c    20
pN2a    17
pN3     14
pN3b    12
Name: count, dtype: int64


In [47]:
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print(missing)

days_to_metastasis_4                  332
metastasis_4_locations                332
days_to_metastasis_3                  330
metastasis_3_locations                330
progress_2                            323
days_to_progress_2                    323
metastasis_2_locations                321
days_to_metastasis_2                  321
days_to_metastasis_1                  283
metastasis_1_locations                283
days_to_progress_1                    263
days_to_recurrence                    259
adjuvant_systemic_therapy_modality    186
perinodal_invasion                    103
adjuvant_treatment_intent              82
adjuvant_radiotherapy_modality         81
infiltration_depth_in_mm               48
number_of_positive_lymph_nodes         27
closest_resection_margin_in_cm         23
primarily_metastasis                   16
smoking_status                          5
histologic_type                         1
first_treatment_modality                0
adjuvant_systemic_therapy         

In [48]:
df_model = df[
[
    'age_at_initial_diagnosis',
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'days_to_first_treatment',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status',
    'infiltration_depth_in_mm',
    'HPV'
]
].copy()

In [49]:
print(df_model.isnull().sum())

age_at_initial_diagnosis                0
sex                                     0
smoking_status                          5
primarily_metastasis                   16
first_treatment_intent                  0
first_treatment_modality                0
days_to_first_treatment                 0
adjuvant_treatment_intent              82
adjuvant_radiotherapy                   0
adjuvant_radiotherapy_modality         81
adjuvant_systemic_therapy               0
adjuvant_systemic_therapy_modality    186
adjuvant_radiochemotherapy              0
primary_tumor_site                      0
pT_stage                                0
pN_stage                                0
histologic_type                         1
number_of_positive_lymph_nodes         27
number_of_resected_lymph_nodes          0
perinodal_invasion                    103
lymphovascular_invasion_L               0
vascular_invasion_V                     0
perineural_invasion_Pn                  0
resection_status                  

In [51]:
cat_cols = [
    'sex',
    'smoking_status',
    'primarily_metastasis',
    'first_treatment_intent',
    'first_treatment_modality',
    'adjuvant_treatment_intent',
    'adjuvant_radiotherapy',
    'adjuvant_radiotherapy_modality',
    'adjuvant_systemic_therapy',
    'adjuvant_systemic_therapy_modality',
    'adjuvant_radiochemotherapy',
    'primary_tumor_site',
    'pT_stage',
    'pN_stage',
    'histologic_type',
    'perinodal_invasion',
    'lymphovascular_invasion_L',
    'vascular_invasion_V',
    'perineural_invasion_Pn',
    'resection_status'
]

for col in cat_cols:
    df_model[col] = df_model[col].fillna(
        df_model[col].mode()[0]
    )

In [52]:
num_cols = [
    'age_at_initial_diagnosis',
    'days_to_first_treatment',
    'number_of_positive_lymph_nodes',
    'number_of_resected_lymph_nodes',
    'infiltration_depth_in_mm'
]

for col in num_cols:
    df_model[col] = df_model[col].fillna(
        df_model[col].median()
    )

In [53]:
print(df_model.isnull().sum())

age_at_initial_diagnosis              0
sex                                   0
smoking_status                        0
primarily_metastasis                  0
first_treatment_intent                0
first_treatment_modality              0
days_to_first_treatment               0
adjuvant_treatment_intent             0
adjuvant_radiotherapy                 0
adjuvant_radiotherapy_modality        0
adjuvant_systemic_therapy             0
adjuvant_systemic_therapy_modality    0
adjuvant_radiochemotherapy            0
primary_tumor_site                    0
pT_stage                              0
pN_stage                              0
histologic_type                       0
number_of_positive_lymph_nodes        0
number_of_resected_lymph_nodes        0
perinodal_invasion                    0
lymphovascular_invasion_L             0
vascular_invasion_V                   0
perineural_invasion_Pn                0
resection_status                      0
infiltration_depth_in_mm              0


In [54]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    df_model[col] = LabelEncoder().fit_transform(
        df_model[col].astype(str)
    )

In [55]:
X = df_model.drop(columns=['HPV'])
y = df_model['HPV']

In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [57]:
print("Train:")
print(y_train.value_counts())

print("\nTest:")
print(y_test.value_counts())

Train:
HPV
0    152
1    113
Name: count, dtype: int64

Test:
HPV
0    39
1    28
Name: count, dtype: int64


In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [59]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [60]:
predicted = model.predict(X_test)

probabilities = model.predict_proba(
    X_test
)[:,1]

In [61]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

print(classification_report(
    y_test,
    predicted,
    digits=4
))

cm = confusion_matrix(
    y_test,
    predicted
)

print(cm)

bal_acc = balanced_accuracy_score(
    y_test,
    predicted
)

f1 = f1_score(
    y_test,
    predicted
)

auc = roc_auc_score(
    y_test,
    probabilities
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

              precision    recall  f1-score   support

           0     0.7674    0.8462    0.8049        39
           1     0.7500    0.6429    0.6923        28

    accuracy                         0.7612        67
   macro avg     0.7587    0.7445    0.7486        67
weighted avg     0.7602    0.7612    0.7578        67

[[33  6]
 [10 18]]
Balanced Accuracy: 0.7445
F1-score: 0.6923
AUC: 0.7921
